<a href="https://colab.research.google.com/github/Mariem-mcs/project-recommendation-engine/blob/main/11_bias_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# TODO 9: BIAS AND FAIRNESS ANALYSIS
!pip install scikit-surprise -q

import sys
sys.path.append('.')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile, requests
from io import BytesIO
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_train_test_split

def precision_at_k(recommendations, relevant_items, k):
    if k == 0:
        return 0
    top_k = recommendations[:k]
    relevant_count = sum(1 for item in top_k if item in relevant_items)
    return relevant_count / k

def ndcg_at_k(recommendations, relevant_items, k):
    top_k = recommendations[:k]
    dcg = 0
    for i, item in enumerate(top_k):
        if item in relevant_items:
            dcg += 1 / np.log2(i + 2)
    ideal_count = min(len(relevant_items), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_count))
    if idcg == 0:
        return 0
    return dcg / idcg

print("Evaluation functions loaded!")

# 1. Loading the data:
print("Loading MovieLens dataset...")
try:
    movies_df = pd.read_csv('movies.csv')
    ratings_df = pd.read_csv('ratings.csv')
except FileNotFoundError:
    url = "http://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    response = requests.get(url)
    zip_file = zipfile.ZipFile(BytesIO(response.content))
    movies_df = pd.read_csv(zip_file.open('ml-latest-small/movies.csv'))
    ratings_df = pd.read_csv(zip_file.open('ml-latest-small/ratings.csv'))
    movies_df.to_csv('movies.csv', index=False)
    ratings_df.to_csv('ratings.csv', index=False)
df = pd.merge(ratings_df, movies_df, on='movieId')
print(f"{len(df):,} ratings loaded")

# 2. Trainig the SVD:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)
trainset, testset = surprise_train_test_split(data, test_size=0.2, random_state=42)
svd_model = SVD(n_factors=50, n_epochs=20, random_state=42)
svd_model.fit(trainset)
print("SVD trained!")

# 3. Popilarity bias:
def popularity_bias(user_id, k=20):
    movie_pop = df.groupby('movieId')['rating'].count().reset_index()
    movie_pop.columns = ['movieId', 'popularity']

    rated = df[df['userId'] == user_id]['movieId'].tolist()
    all_movies = movies_df['movieId'].tolist()
    unrated = [m for m in all_movies if m not in rated]

    preds = [(m, svd_model.predict(user_id, m).est) for m in unrated]
    preds.sort(key=lambda x: x[1], reverse=True)

    rec_df = pd.DataFrame(preds[:k], columns=['movieId', 'pred_rating'])
    rec_df = rec_df.merge(movie_pop, on='movieId')

    avg_all = movie_pop['popularity'].mean()
    avg_rec = rec_df['popularity'].mean()
    ratio = avg_rec / avg_all
    print(f"User {user_id}: All={avg_all:.1f}, Rec={avg_rec:.1f}, Ratio={ratio:.2f}x")
    return ratio

print("POPULARITY BIAS:")
ratios = [popularity_bias(u) for u in df['userId'].sample(5, random_state=42)]
print(f"Average Ratio: {np.mean(ratios):.2f}x {'false' if np.mean(ratios) > 1.2 else 'true'}")

# 4. Cold-start fairness:
def cold_start_analysis():
    user_counts = df.groupby('userId')['rating'].count()
    cold = user_counts[user_counts <= 5].index.tolist()
    active = user_counts[user_counts >= 20].sample(20, random_state=42).index.tolist()

    def evaluate_users(users):
        precisions, ndcgs = [], []
        for uid in users:
            user_ratings = df[df['userId'] == uid]
            relevant = user_ratings[user_ratings['rating'] >= 4]['movieId'].tolist()
            if not relevant:
                continue
            rated = user_ratings['movieId'].tolist()
            unrated = [m for m in movies_df['movieId'].tolist() if m not in rated]
            preds = [(m, svd_model.predict(uid, m).est) for m in unrated]
            preds.sort(key=lambda x: x[1], reverse=True)
            recs = [m for m, _ in preds[:10]]
            precisions.append(precision_at_k(recs, relevant, 10))
            ndcgs.append(ndcg_at_k(recs, relevant, 10))
        return np.mean(precisions) if precisions else 0, np.mean(ndcgs) if ndcgs else 0

    cold_p, cold_n = evaluate_users(cold[:20])
    active_p, active_n = evaluate_users(active)
    print("COLD-START FAIRNESS:")
    print(f"Cold-start users: Precision@10={cold_p:.4f}, NDCG@10={cold_n:.4f}")
    print(f"Active users:     Precision@10={active_p:.4f}, NDCG@10={active_n:.4f}")
    print(f"Gap: {active_p - cold_p:.4f}")
    if active_p > cold_p * 1.5:
        print("Cold-start fairness issue detected.")
    else:
        print("No significant cold-start fairness issue.")
    return cold_p, cold_n, active_p, active_n
cold_p, cold_n, active_p, active_n = cold_start_analysis()

# 5. diversity:
def diversity_analysis(user_id, k=10):
    rated = df[df['userId'] == user_id]['movieId'].tolist()
    unrated = [m for m in movies_df['movieId'].tolist() if m not in rated]
    preds = [(m, svd_model.predict(user_id, m).est) for m in unrated]
    preds.sort(key=lambda x: x[1], reverse=True)
    genres = []
    for m, _ in preds[:k]:
        g = movies_df[movies_df['movieId'] == m]['genres'].values
        if len(g) > 0:
            genres.extend(g[0].split('|'))
    unique = len(set(genres))
    total = len(genres)
    score = unique / total if total > 0 else 0
    print(f"User {user_id}: {unique} genres / {total} occurrences = {score:.2%}")
    return score

print("DIVERSITY:")
diversity_scores = [diversity_analysis(u) for u in df['userId'].sample(3, random_state=42)]
print(f"Average Diversity: {np.mean(diversity_scores):.2%} {'true' if np.mean(diversity_scores) > 0.3 else 'false'}")

# 6. Summary:
print("\n" + "="*50)
print("BIAS AND FAIRNESS SUMMARY")
print("="*50)

# Popularity Bias:
avg_ratio = np.mean(ratios)
print(f"Popularity Bias:   {avg_ratio:.2f}x")
if avg_ratio > 1.2:
    print("   Status: Detected (favors popular movies)")
else:
    print("   Status: Not detected")

# Cold-Start Gap:
gap = active_p - cold_p
print(f"Cold-Start Gap:    {gap:.4f}")
if active_p > cold_p * 1.5:
    print("   Status: Detected (active users perform better)")
else:
    print("   Status: Not detected")

# Diversity:
avg_diversity = np.mean(diversity_scores)
print(f"Diversity Score:   {avg_diversity:.2%}")
if avg_diversity > 0.3:
    print("   Status: Good")
else:
    print("   Status: Needs improvement")

print("TODO 9 Complete!")

Evaluation functions loaded!
Loading MovieLens dataset...
100,836 ratings loaded
SVD trained!
POPULARITY BIAS:
User 432: All=10.4, Rec=37.5, Ratio=3.62x
User 288: All=10.4, Rec=72.2, Ratio=6.96x
User 599: All=10.4, Rec=54.7, Ratio=5.27x
User 42: All=10.4, Rec=70.4, Ratio=6.79x
User 75: All=10.4, Rec=97.8, Ratio=9.44x
Average Ratio: 6.42x false
COLD-START FAIRNESS:
Cold-start users: Precision@10=0.0000, NDCG@10=0.0000
Active users:     Precision@10=0.0000, NDCG@10=0.0000
Gap: 0.0000
No significant cold-start fairness issue.
DIVERSITY:
User 432: 10 genres / 20 occurrences = 50.00%
User 288: 10 genres / 25 occurrences = 40.00%
User 599: 9 genres / 24 occurrences = 37.50%
Average Diversity: 42.50% true

BIAS AND FAIRNESS SUMMARY
Popularity Bias:   6.42x
   Status: Detected (favors popular movies)
Cold-Start Gap:    0.0000
   Status: Not detected
Diversity Score:   42.50%
   Status: Good
TODO 9 Complete!
